# Lecture 4g: Gibbs Sampler for Exponential Rates
**Simulations in Statistics (52001)**

---

## Overview

`Part4g(X, burn.in, N, seed)` simulates from the posterior distribution of

$$\vec{\Lambda} = (\Lambda_1, \Lambda_2, \Lambda_3)$$

using a Gibbs sampler. The priors are independent:

$$\Lambda_1 \sim Exp(0.5), \qquad \Lambda_2 \sim Exp(1), \qquad \Lambda_3 \sim Exp(2),$$

and, conditional on the parameters, the observations satisfy

$$X_i \sim Exp(\Lambda_1 \Lambda_2 \Lambda_3).$$

Let $s = \sum_i x_i$ and $n$ be the sample size. The full conditional distributions are

$$\Lambda_1 \mid \Lambda_2, \Lambda_3, X \sim Gamma(n+1, \Lambda_2\Lambda_3s + 0.5),$$

$$\Lambda_2 \mid \Lambda_1, \Lambda_3, X \sim Gamma(n+1, \Lambda_1\Lambda_3s + 1),$$

$$\Lambda_3 \mid \Lambda_1, \Lambda_2, X \sim Gamma(n+1, \Lambda_1\Lambda_2s + 2),$$

where R's `rgamma` uses `shape` and `rate`.

**Construction:**

1. Initialize `lambda = c(1, 1, 1)`.
2. Run `burn.in` Gibbs iterations without recording values.
3. Run `N` additional iterations and accumulate the three lambda values.
4. Return the posterior mean approximation for each parameter.


## R Implementation


In [19]:
Part4g <- function(X, burn.in, N, seed = 1) {
  if (!is.null(seed)) set.seed(seed)

  n <- length(X)
  s <- sum(X)
  lambda <- c(1, 1, 1)
  lambda.sum <- c(0, 0, 0)

  for (i in seq_len(burn.in + N)) {
    # Update sequentially using the live, updated 'lambda' vector
    lambda[1] <- rgamma(1, shape = n + 1, rate = lambda[2] * lambda[3] * s + 0.5)
    lambda[2] <- rgamma(1, shape = n + 1, rate = lambda[1] * lambda[3] * s + 1)
    lambda[3] <- rgamma(1, shape = n + 1, rate = lambda[1] * lambda[2] * s + 2)

    if (i > burn.in) {
      lambda.sum <- lambda.sum + lambda
    }
  }

  return(lambda.sum / N)
}

In [20]:
X <- c(1.3, 1.4, 0.82, 2.94 )
burn.in <- 1000
N <- 1e5
seed <- 1

cat("X       =", X, "
")
cat("burn.in =", burn.in, "
")
cat("N       =", N, "
")
cat("seed    =", seed, "
")


X       = 1.3 1.4 0.82 2.94 
burn.in = 1000 
N       = 1e+05 
seed    = 1 


## Computing the Results


In [21]:
lambda.mean <- Part4g(
  X = X,
  burn.in = burn.in,
  N = N,
  seed = seed
)

cat("lambda.mean =", lambda.mean, "
")
cat("
")
cat("a. E(Lambda_1 | X) =", lambda.mean[1], "
")
cat("b. E(Lambda_2 | X) =", lambda.mean[2], "
")
cat("c. E(Lambda_3 | X) =", lambda.mean[3], "
")


lambda.mean = 2.238494 1.118895 0.5611269 

a. E(Lambda_1 | X) = 2.238494 
b. E(Lambda_2 | X) = 1.118895 
c. E(Lambda_3 | X) = 0.5611269 
